In [1]:
from google.oauth2 import service_account
from google.cloud import bigquery
import pandas as pd

key_path = "patbis-b85b8ca5802e.json"
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project="patbis")

In [ ]:
"""
geocode_patstat_bq.py
─────────────────────────────────────────────────────────────────────────────
Geocodes persons from patbis.fromPATSTAT2025.tls206_person using Nominatim,
writing results back to BigQuery incrementally.

Source  : patbis.fromPATSTAT2025.tls206_person_with_address
Output  : patbis.fromPATSTAT2025.tls206_person_with_address_geocoded  (auto-created)

Resume  : on restart, already-geocoded person_id rows are skipped via a
          NOT EXISTS subquery — no local state needed.
Retries : BQ query failures use exponential backoff (up to MAX_RETRIES).
          Nominatim connection failures sleep 5 min then retry.
─────────────────────────────────────────────────────────────────────────────
"""

import os
import time
import logging
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import GoogleAPIError
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
PROJECT_ID      = "patbis"
KEY_PATH        = os.path.join(os.path.dirname(__file__), "patbis-b85b8ca5802e.json") \
                  if "__file__" in globals() else "patbis-b85b8ca5802e.json"
SOURCE_TABLE    = "patbis.fromPATSTAT2025.tls206_person_with_address"
OUTPUT_DATASET  = "fromPATSTAT2025"
OUTPUT_TABLE    = "tls206_person_with_address_geocoded"
OUTPUT_TABLE_FQ = f"{PROJECT_ID}.{OUTPUT_DATASET}.{OUTPUT_TABLE}"

BATCH_SIZE      = 50        # rows per Nominatim batch
COOLDOWN_EVERY  = 10        # batches between cooldown pauses
COOLDOWN_SECS   = 60        # cooldown duration
CHECKPOINT_EVERY= 20        # batches between progress logs
MAX_RETRIES     = 5         # BQ query retries
USER_AGENTS     = ["patstat_geo_v1", "patstat_geo_v2", "patstat_geo_v3"]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── BQ helpers ────────────────────────────────────────────────────────────────
def get_client():
    """Create a BQ client. Called once at startup and after reconnection."""
    credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
    return bigquery.Client(credentials=credentials, project=PROJECT_ID)


def ensure_output_table(client):
    """Create output table if it doesn't exist yet."""
    schema = [
        bigquery.SchemaField("person_id",          "STRING"),
        bigquery.SchemaField("person_address",     "STRING"),
        bigquery.SchemaField("person_ctry_code",   "STRING"),
        bigquery.SchemaField("latest_filing_date", "STRING"),
        bigquery.SchemaField("latitude",           "FLOAT"),
        bigquery.SchemaField("longitude",          "FLOAT"),
        bigquery.SchemaField("geocode_level",      "STRING"),  # full/last3/last2/last1/failed/empty
        bigquery.SchemaField("geocoded_at",        "TIMESTAMP"),
    ]
    table_ref = bigquery.Table(OUTPUT_TABLE_FQ, schema=schema)
    client.create_table(table_ref, exists_ok=True)
    log.info(f"Output table ready: {OUTPUT_TABLE_FQ}")


def fetch_batch_with_retry(client, offset, batch_size, retries=MAX_RETRIES):
    """
    Pull one batch of ungeocoded rows from BQ.
    Uses NOT EXISTS so already-written rows are automatically skipped.
    Exponential backoff on failure.
    """
    query = f"""
        SELECT
            CAST(p.person_id AS STRING) AS person_id,
            p.person_address,
            p.person_ctry_code,
            p.latest_filing_date
        FROM `{SOURCE_TABLE}` p
        WHERE NOT EXISTS (
            SELECT 1
            FROM `{OUTPUT_TABLE_FQ}` g
            WHERE g.person_id = CAST(p.person_id AS STRING)
        )
        ORDER BY p.person_id
        LIMIT  {batch_size}
        OFFSET {offset}
    """
    for attempt in range(1, retries + 1):
        try:
            df = client.query(query).to_dataframe()
            return df
        except GoogleAPIError as e:
            wait = 2 ** attempt
            log.warning(f"BQ error (attempt {attempt}/{retries}): {e} — retrying in {wait}s")
            time.sleep(wait)
        except Exception as e:
            log.warning(f"Unexpected error (attempt {attempt}/{retries}): {e} — retrying in {wait}s")
            wait = 2 ** attempt
            time.sleep(wait)
    raise RuntimeError(f"BQ fetch failed after {retries} retries at offset {offset}")


def count_remaining(client):
    """Quick count of rows not yet geocoded."""
    query = f"""
        SELECT COUNT(*) AS n
        FROM `{SOURCE_TABLE}` p
        WHERE NOT EXISTS (
            SELECT 1 FROM `{OUTPUT_TABLE_FQ}` g
            WHERE g.person_id = CAST(p.person_id AS STRING)
        )
    """
    result = client.query(query).to_dataframe()
    return int(result['n'].iloc[0])


def write_batch_to_bq(client, df):
    """Append a geocoded batch to the output table."""
    df = df.copy()
    df['geocoded_at'] = pd.Timestamp.utcnow()

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        schema_update_options=[bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION],
    )
    job = client.load_table_from_dataframe(df, OUTPUT_TABLE_FQ, job_config=job_config)
    job.result()   # wait for completion


# ── Country-hint detection ────────────────────────────────────────────────────
def add_country_hint(address, ctry_code=None):
    if not isinstance(address, str):
        return address
    a = address.upper()
    hints = {
        'JAPAN':   ['CHOME', '-SHI', '-KU', '-KEN', 'KABUSHIKI'],
        'BELGIUM': ['W.VL.', 'O.VL.', 'ANTW.', 'ZEDELGEM', 'AARTRIJKE'],
        'USA':     ['CONN.', ' CT ', ' NY ', ' CA ', ' MA ', ' TX '],
    }
    for country, keywords in hints.items():
        if country not in a and any(k in a for k in keywords):
            return address.strip() + f', {country}'
    if ctry_code and isinstance(ctry_code, str) and ctry_code.upper() not in a:
        return address.strip() + f', {ctry_code.upper()}'
    return address


# ── Progressive fallback geocoder ─────────────────────────────────────────────
def geocode_with_fallback(row, geocode_fn):
    address = row.get('person_address')
    ctry    = row.get('person_ctry_code')

    if not isinstance(address, str) or not address.strip():
        return pd.Series([None, None, 'empty'], index=['latitude', 'longitude', 'geocode_level'])

    address = add_country_hint(address, ctry)
    parts   = [p.strip() for p in address.split(',')]

    attempts = [
        address,
        ', '.join(parts[-3:]) if len(parts) >= 3 else None,
        ', '.join(parts[-2:]) if len(parts) >= 2 else None,
        parts[-1] if parts else None,
    ]
    attempts = [a for a in attempts if a]
    labels   = ['full', 'last3', 'last2', 'last1']

    for i, attempt in enumerate(attempts):
        try:
            result = geocode_fn(attempt)
            if result:
                return pd.Series(
                    [result.latitude, result.longitude, labels[i]],
                    index=['latitude', 'longitude', 'geocode_level']
                )
        except Exception as e:
            if "timed out" in str(e).lower() or "connection" in str(e).lower():
                log.warning("Nominatim connection issue — sleeping 5 min...")
                time.sleep(300)

    return pd.Series([None, None, 'failed'], index=['latitude', 'longitude', 'geocode_level'])


def make_geocoder(agent):
    geolocator = Nominatim(user_agent=agent)
    return RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1.5,
        max_retries=5,
        error_wait_seconds=15.0,
        swallow_exceptions=True,
    )


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    client = get_client()
    ensure_output_table(client)

    remaining = count_remaining(client)
    log.info(f"Rows remaining to geocode: {remaining:,}")
    if remaining == 0:
        log.info("Nothing to do — exiting.")
        return

    batch_num = 0
    total_written = 0

    while True:
        # ── fetch next batch ──────────────────────────────────────────────────
        # offset=0 always because NOT EXISTS shrinks the pool after each write
        try:
            batch = fetch_batch_with_retry(client, offset=0, batch_size=BATCH_SIZE)
        except RuntimeError as e:
            log.error(f"Fatal BQ fetch error: {e}")
            break

        if batch.empty:
            log.info("No more rows — pipeline complete.")
            break

        log.info(f"Batch {batch_num + 1} — {len(batch)} rows")

        # ── cooldown ──────────────────────────────────────────────────────────
        if batch_num > 0 and batch_num % COOLDOWN_EVERY == 0:
            log.info(f"  Cooling down {COOLDOWN_SECS}s...")
            time.sleep(COOLDOWN_SECS)

        # ── geocode ───────────────────────────────────────────────────────────
        geocode_fn = make_geocoder(USER_AGENTS[batch_num % len(USER_AGENTS)])
        tqdm.pandas(desc="  Geocoding", leave=False)

        results = batch.progress_apply(
            lambda row: geocode_with_fallback(row, geocode_fn), axis=1
        )
        batch = pd.concat([batch, results], axis=1)

        # ── write to BQ ───────────────────────────────────────────────────────
        write_batch_to_bq(client, batch)
        total_written += len(batch)
        batch_num += 1

        # ── checkpoint ────────────────────────────────────────────────────────
        if batch_num % CHECKPOINT_EVERY == 0:
            try:
                remaining = count_remaining(client)
                done_q = f"SELECT COUNT(*) AS n FROM `{OUTPUT_TABLE_FQ}`"
                done   = int(client.query(done_q).to_dataframe()['n'].iloc[0])
                pct    = done / (done + remaining) * 100 if (done + remaining) > 0 else 0
                log.info(f"── Checkpoint: {done:,} done | {remaining:,} remaining ({pct:.1f}%) ──")
            except Exception as e:
                log.warning(f"Checkpoint query failed (non-fatal): {e}")

    # ── final summary ─────────────────────────────────────────────────────────
    log.info(f"\nAll done — {total_written:,} rows written this session.")
    try:
        summary_q = f"""
            SELECT geocode_level, COUNT(*) AS n
            FROM `{OUTPUT_TABLE_FQ}`
            GROUP BY geocode_level
            ORDER BY n DESC
        """
        summary = client.query(summary_q).to_dataframe()
        log.info(f"\n{summary.to_string(index=False)}")
    except Exception as e:
        log.warning(f"Summary query failed: {e}")


if __name__ == "__main__":
    main()